In [1]:
# !pip install langchain langchain-openai langchain-google-genai

In [38]:
MODEL_NAME="gemma-4-31b-it"

In [39]:
from dotenv import load_dotenv
load_dotenv()
import os
from pprint import pprint

In [40]:
from langchain_core.prompts import ChatPromptTemplate

In [41]:
GEMINI_API = os.getenv("GEMINI_API")
print(len(GEMINI_API))

53


# __1. Các thành phần cốt lõi của Langchain__

## 1.1. ChatPromptTemplate
Đây là “form điền vào chỗ trống”. Thay vì viết cứng cả câu, ta để một biến {...} cho phần thay đổi:

In [42]:
prompt = ChatPromptTemplate.from_messages([
    {"role": "system","content": "Bạn là 1 trợ lý tìm kiếm thông tin"},
    {"role": "user", "content": "Hãy tìm thông tin về {text}"} # biến {text} sẽ được điền lúc chạy nên ta có thể tái sử dụng
])

## 1.2. LLM

In [43]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [44]:
model = ChatGoogleGenerativeAI(model=MODEL_NAME, api_key=GEMINI_API)

## 1.3. StrOutputParser
Mô hình chat trả về 1 đối tượng `message` chứ không phải chuỗi thuần. `StrOutputParser` sẽ lấy đúng phần text trong đó ra

In [45]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## 1.4. Nối lại bằng LCEL: `prompt | llm | parser`
`LCEL (LangChain Expression Language)` cho phép nối các thành phần bằng `toán tử pipe |` - giống ống nước, dữ liệu chảy từ trái sang phải

In [46]:
chain = prompt | model | parser

In [47]:
res = chain.invoke({"text": "tóm tắt về sự kiện 11/9 tại NYC"})

In [48]:
pprint(res)

('Sự kiện ngày 11 tháng 9 năm 2001 (thường được gọi là **11/9**) là một loạt '
 'các cuộc tấn công khủng bố phối hợp nhắm vào Hoa Kỳ, trong đó thành phố New '
 'York (NYC) là tâm điểm của sự tàn phá.\n'
 '\n'
 'Dưới đây là tóm tắt chi tiết về sự kiện này tại New York City:\n'
 '\n'
 '### 1. Tổng quan sự kiện\n'
 '*   **Thời gian:** Sáng ngày 11 tháng 9 năm 2001.\n'
 '*   **Thủ phạm:** Tổ chức khủng bố Hồi giáo cực đoan **Al-Qaeda**, do Osama '
 'bin Laden cầm đầu.\n'
 '*   **Phương thức:** Bắt cóc 4 máy bay thương mại để biến chúng thành "tên '
 'lửa bay" lao vào các mục tiêu chiến lược.\n'
 '\n'
 '### 2. Diễn biến tại New York City\n'
 'Tại NYC, hai chiếc máy bay đã bị không tặc điều khiển lao thẳng vào tòa tháp '
 'đôi của **Trung tâm Thương mại Thế giới (World Trade Center - WTC)**:\n'
 '\n'
 '*   **Chiếc máy bay thứ nhất (American Airlines Flight 11):** Lúc 8:46 sáng, '
 'lao vào Tháp Bắc (North Tower).\n'
 '*   **Chiếc máy bay thứ hai (United Airlines Flight 175):** Lúc 9:03 sáng,

# __2. Thêm memory (ConversationBufferMemory) cho agent__

Loại memory đơn giản nhất của LangChain classic là `ConversationBufferMemory`: nó lưu toàn bộ lịch sử hội thoại, không cắt xén gì. “Buffer” = cái đệm chứa tất cả.

## 2.1. Các loại memory
|Loại memory|Ý nghĩa|Ưu điểm|Nhược điểm|
|-|-|-|-|
|Buffer|Lưu toàn bộ lịch sử, nguyên văn|Nhớ đầy đủ, chính xác|Phình to, tốn token, dễ vượt context|
|Windows|Chỉ lưu k lượt gần nhất|Gọn, chi phí ổn|Dễ quên nếu vượt khỏi windows|
|Summary|Bản tóm tắt hội thoại do LLM viết|Nén rất tốt cho hội thoại dài|Tốn thêm 1 request để tóm tắt và có thể mất chi tiết|

## 2.2. Cách làm hiện đại
+ Tự quản lý 1 danh sách message `(chat history)` và bọc chain bằng `RunnableWithMessageHistory` hoặc dùng langgraph
+ Nếu muốn dùng `summary` thì sẽ dùng `LangGraph`

In [50]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

In [61]:
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"), # Thêm cái này để đánh dấu là history
    {"role": "user", "content": "{input}"} # biến {text} sẽ được điền lúc chạy nên ta có thể tái sử dụng
])

In [62]:
chain = prompt | model | parser

In [63]:
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [64]:
conversation = RunnableWithMessageHistory(
    chain, 
    get_session_history, # Hàm lấy history
    input_messages_key="input", 
    history_messages_key="history"
)

e:\Mine\Code\LearnAI\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [65]:
config={"configurable": {"session_id": "abc"}}

In [ ]:
response = conversation.invoke({"input": "Xin chào, tôi là Q"}, config=config)
pprint(response)

Xin chào Q! Rất vui được làm quen với bạn. 

Mình là một trợ lý AI. Hôm nay của bạn thế nào? Có điều gì bạn muốn chia sẻ hoặc cần mình hỗ trợ không? 😊


In [ ]:
res = conversation.invoke({"input": "Tôi là ai?"}, config=config)
pprint(res)

Dựa trên những gì bạn vừa chia sẻ với mình ở câu trước, bạn là **Q**. 😊

Vì mình là một trí tuệ nhân tạo, mình không biết danh tính thật sự, khuôn mặt hay cuộc đời của bạn ngoài đời thực trừ khi bạn kể cho mình nghe. Đối với mình, bạn là một người bạn đang trò chuyện cùng! 

Bạn có muốn chia sẻ thêm điều gì về bản thân mình không?
